# Bikeability Index & Usage Modeling — Reference-Style Notebook

This notebook scaffolds a workflow to build a Bikeability Index (BI) from OSM using OSMnx and (optionally) model usage.

In [2]:
# Environment check (edit your env separately)
import os, warnings, json, math
import numpy as np
import pandas as pd
import geopandas as gpd
import osmnx as ox
from shapely.geometry import box, Point, Polygon
from shapely.ops import unary_union

ox.settings.use_cache = True
ox.settings.log_console = False

print("osmnx:", ox.__version__)
print("geopandas:", gpd.__version__)


osmnx: 2.0.3
geopandas: 1.0.1


In [3]:

# CONFIG
PLACE = "Munich, Germany"
CRS_METRIC = 3857
CELL_M = 2000


In [4]:

import osmnx as ox
boundary = ox.geocode_to_gdf(PLACE).to_crs(4326)
poly = boundary.geometry.unary_union
boundary_m = boundary.to_crs(3857)
display(boundary.head(1))


C:\Users\valeriia.sailaonova\AppData\Local\Temp\ipykernel_35376\3121639198.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  poly = boundary.geometry.unary_union


,geometry,bbox_west,bbox_south,bbox_east,bbox_north,place_id,osm_type,osm_id,lat,lon,class,type,place_rank,importance,addresstype,name,display_name
0,"MULTIPOLYGON (((11.36078 48.15807, 11.36085 48...",11.360777,48.061624,11.72291,48.248116,117173815,relation,62428,48.137108,11.575382,boundary,administrative,12,0.799006,city,Munich,"Munich, Bavaria, Germany"


In [5]:

# Robust custom filter
cf = (
    '["highway"~"motorway|trunk|primary|secondary|tertiary|residential|'
    'unclassified|living_street|service|track|pedestrian|footway|path|cycleway"]'
    '["area"!~"yes"]'
)
G = ox.graph_from_polygon(poly, custom_filter=cf, simplify=True)
nodes, edges = ox.graph_to_gdfs(G, nodes=True, edges=True, fill_edge_geometry=True)
edges = edges.to_crs(CRS_METRIC)
boundary_m = boundary_m.to_crs(CRS_METRIC)
print("nodes", len(nodes), "edges", len(edges))
edges.head(2)


nodes 163020 edges 421466


osmid       highway maxspeed                 name  \
u      v          key                                                        
128236 1399723741 0    4055519  unclassified       30          Pirolstraße   
       1097240022 0    4872976     secondary       50  Lochhausener Straße   

                       oneway reversed     length lanes      ref  \
u      v          key                                              
128236 1399723741 0     False     True   7.215758   NaN      NaN   
       1097240022 0     False    False  17.320692     2  St 2345   

                                                                geometry  \
u      v          key                                                      
128236 1399723741 0    LINESTRING (1268519.038 6136828.109, 1268515.3...   
       1097240022 0    LINESTRING (1268519.038 6136828.109, 1268544.2...   

                      bridge width access service tunnel junction est_width  \
u      v          key                                                         
128236 1399723741 0      NaN   NaN    NaN     NaN    NaN      NaN       NaN   
       1097240022 0      NaN   NaN    NaN     NaN    NaN      NaN       NaN   

                      area  
u      v          key       
128236 1399723741 0    NaN  
       1097240022 0    NaN

In [6]:

import pandas as pd
import numpy as np

edges["hw"] = edges["highway"].astype(str)
ROADCLASS_MAIN = {"motorway","trunk","primary","primary_link","secondary","secondary_link"}
ROADCLASS_MINOR = {"tertiary","tertiary_link","residential","living_street","unclassified"}
ROADCLASS_PATHS = {"cycleway","path","footway","pedestrian","track","service"}

def classify_roadclass(hw):
    if hw in ROADCLASS_MAIN: return "main"
    if hw in ROADCLASS_MINOR: return "minor"
    if hw in ROADCLASS_PATHS: return "pathlike"
    return "other"

edges["roadclass"] = edges["hw"].apply(classify_roadclass)
cycle_cols = [c for c in edges.columns if c.startswith("cycleway")]
edges["cycle_fac_any"] = edges[cycle_cols].notna().any(axis=1) if cycle_cols else False
edges["bicycle_road"] = edges.get("bicycle_road", pd.Series(index=edges.index)).astype(str).eq("yes")
edges["is_main"] = edges["roadclass"].eq("main")
edges["main_with_fac"] = edges["is_main"] & (edges["cycle_fac_any"] | edges["bicycle_road"] | edges["hw"].eq("cycleway"))
edges["length_m"] = edges.geometry.length
edges = edges[edges["length_m"]>0].copy()
edges.head(2)


osmid       highway maxspeed                 name  \
u      v          key                                                        
128236 1399723741 0    4055519  unclassified       30          Pirolstraße   
       1097240022 0    4872976     secondary       50  Lochhausener Straße   

                       oneway reversed     length lanes      ref  \
u      v          key                                              
128236 1399723741 0     False     True   7.215758   NaN      NaN   
       1097240022 0     False    False  17.320692     2  St 2345   

                                                                geometry  ...  \
u      v          key                                                     ...   
128236 1399723741 0    LINESTRING (1268519.038 6136828.109, 1268515.3...  ...   
       1097240022 0    LINESTRING (1268519.038 6136828.109, 1268544.2...  ...   

                      junction est_width area            hw roadclass  \
u      v          key                                                   
128236 1399723741 0        NaN       NaN  NaN  unclassified     minor   
       1097240022 0        NaN       NaN  NaN     secondary      main   

                      cycle_fac_any bicycle_road is_main main_with_fac  \
u      v          key                                                    
128236 1399723741 0           False        False   False         False   
       1097240022 0           False        False    True         False   

                        length_m  
u      v          key             
128236 1399723741 0    10.833662  
       1097240022 0    26.005107  

[2 rows x 25 columns]

In [7]:

from shapely.geometry import box

minx, miny, maxx, maxy = boundary_m.total_bounds
polys = []
x = minx
while x < maxx:
    y = miny
    while y < maxy:
        polys.append(box(x, y, x+CELL_M, y+CELL_M))
        y += CELL_M
    x += CELL_M
grid = gpd.GeoDataFrame(geometry=polys, crs=boundary_m.crs)
grid = gpd.overlay(grid, boundary_m[["geometry"]], how="intersection").reset_index(drop=True)
grid["cell_id"] = grid.index.astype(int)
print("cells:", len(grid))
grid.head(1)


cells: 221


,geometry,cell_id
0,"POLYGON ((1266675.911 6133113.069, 1266675.911...",0


In [8]:
import numpy as np
import pandas as pd
import geopandas as gpd

def clip_edges_sum_length(lines_gdf, poly, mask=None):
    """
    Total length (float) of lines_gdf (optionally masked) inside poly.
    Robust to empty intersections and overlay/index issues.
    """
    sub = lines_gdf.loc[mask].copy() if mask is not None else lines_gdf
    if sub.empty:
        return 0.0

    sub = sub.reset_index(drop=True)
    poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs=sub.crs).reset_index(drop=True)

    try:
        clip = gpd.overlay(sub, poly_gdf, how="intersection", keep_geom_type=True)
        return float(clip.length.sum()) if not clip.empty else 0.0
    except Exception:
        inter = sub.geometry.intersection(poly)
        return float(inter.length.sum()) if len(inter) else 0.0

def share_to_score(share_pct, bins=(0,20,40,60,80,100), scores=(0,2,4,6,8,10)):
    """
    Map a percentage to a discrete score. Safe for 0/100 and NaN.
    """
    if share_pct is None or (isinstance(share_pct, float) and np.isnan(share_pct)):
        return np.nan
    x = float(np.clip(share_pct, 0, 100))
    idx = np.searchsorted(bins, x, side="right")
    idx = min(idx, len(scores) - 1)
    return scores[idx]

# --- per-cell computation ---
scores = []
for i, cell_poly in enumerate(grid.geometry):
    total_len = clip_edges_sum_length(edges, cell_poly)

    # Road Class: share of minor + pathlike => higher score
    minor_len = clip_edges_sum_length(
        edges, cell_poly, edges["roadclass"].isin(["minor", "pathlike"])
    )
    RC = share_to_score(0 if total_len == 0 else 100 * minor_len / total_len)

    # Main Road Separation
    main_len = clip_edges_sum_length(edges, cell_poly, edges["is_main"])
    main_fac_len = clip_edges_sum_length(edges, cell_poly, edges["main_with_fac"])
    MRSEP = np.nan if main_len == 0 else share_to_score(100 * main_fac_len / main_len)

    scores.append(dict(cell_id=i, RC=RC, MRSEP=MRSEP))

scores_df = pd.DataFrame(scores).set_index("cell_id")
grid_scores = grid.join(scores_df, on="cell_id")
grid_scores.head(2)

,geometry,cell_id,RC,MRSEP
0,"POLYGON ((1266675.911 6133113.069, 1266675.911...",0,10,NaN
1,"POLYGON ((1266675.911 6135113.069, 1266675.911...",1,10,NaN


In [9]:

# Minimal weights example
w = pd.DataFrame({
    "variable":["RC","MRSEP"],
    "weight":[0.5,0.5]
})
def weighted_index(row, wtable):
    vals=[]; wts=[]
    for _, r in wtable.iterrows():
        v = row.get(r["variable"], np.nan)
        if pd.notna(v):
            vals.append(v*r["weight"]); wts.append(r["weight"])
    return np.nan if not vals else sum(vals)/sum(wts)
grid_scores["BI"] = grid_scores.apply(lambda r: weighted_index(r, w), axis=1)
grid_scores[["cell_id","BI","RC","MRSEP"]].head(5)


,cell_id,BI,RC,MRSEP
0,0,10.0,10,NaN
1,1,10.0,10,NaN
2,2,8.0,8,NaN
3,3,10.0,10,NaN
4,4,5.0,8,2.0


In [10]:
# ---- Compute BI from your existing grid_scores ----
import numpy as np
import pandas as pd
from pathlib import Path

# Set weights for the factor columns that you plan to use.
# Include only the factors you care about; it's fine if some columns aren't present yet.
# Example (adjust numbers to your literature-based weights):
WEIGHTS = {
    "RC":    0.26,  # Road Class
    "MRSEP": 0.21,  # Main Road Separation
    "CN":    0.13,  # Cycling Network (connectivity/directness)
    "DISC":  0.13,  # Cycling Road Discomfort (higher=better, if you computed it that way)
    "QLT":   0.13,  # Cycling Road Quality
    "TOP":   0.14,  # Topography (higher=better if you inverted slope to a score)
    "SOFT":  0.00,  # Soft factors (set to >0 if you added them)
    "AESTH": 0.00,  # Aesthetics (set to >0 if you added it)
}

# Keep only weights for columns that actually exist in grid_scores
present_weights = {k: v for k, v in WEIGHTS.items() if k in grid_scores.columns}

# Normalize over the present weights so they sum to 1
total_w = sum(present_weights.values())
if total_w == 0:
    raise ValueError("All weights are zero or none of the specified factor columns exist in grid_scores.")
present_weights = {k: v / total_w for k, v in present_weights.items()}

def weighted_bi(row, w):
    vals, wts = [], []
    for col, wt in w.items():
        val = row[col]
        if pd.notna(val):
            vals.append(val * wt)
            wts.append(wt)
    return np.nan if not vals else sum(vals) / sum(wts)

grid_scores["BI"] = grid_scores.apply(lambda r: weighted_bi(r, present_weights), axis=1)

# ---- Save to a local GeoPackage you can write to ----
from pathlib import Path
out_dir = Path.cwd() / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "bikeability_reference.gpkg"

# If mixed polygon/multipolygon geometries, promote_to_multi helps avoid driver issues
grid_scores.to_file(out_path, layer="bi_grid", driver="GPKG", engine="fiona", promote_to_multi=True)
print("Saved:", out_path)


Saved: C:\Users\valeriia.sailaonova\outputs\bikeability_reference.gpkg


In [14]:
# === Compact add-on: compute CN, DISC, QLT and merge into grid_scores (robust version) ===
import numpy as np
import pandas as pd
import geopandas as gpd

# 0) Helpers -----------------------------------------------------------------

# Safe length calculator (keeps your previous behavior); uses overlay but tolerates edge cases
def clip_edges_sum_length(lines_gdf, poly, mask=None):
    sub = lines_gdf.loc[mask].copy() if mask is not None else lines_gdf
    if sub.empty:
        return 0.0
    sub = sub.reset_index(drop=True)
    try:
        poly_gdf = gpd.GeoDataFrame(geometry=[poly], crs=sub.crs).reset_index(drop=True)
        # keep_geom_type=False avoids dropping mixed-geometry intersections
        clip = gpd.overlay(sub, poly_gdf, how="intersection", keep_geom_type=False)
        return float(clip.length.sum()) if not clip.empty else 0.0
    except Exception:
        # Fallback: direct per-geometry intersection
        inter = sub.geometry.intersection(poly)
        inter = inter[ inter.notna() & (~inter.is_empty) ]
        return float(inter.length.sum()) if len(inter) else 0.0

# Robust per-cell edge clipping WITHOUT overlay (to avoid index/merge quirks)
def intersect_edges_in_cell(edges_gdf, cell_poly):
    sub = edges_gdf.loc[edges_gdf.geometry.intersects(cell_poly)].copy()
    if sub.empty:
        return sub
    sub = sub.reset_index(drop=True)
    sub["geometry"] = sub.geometry.apply(lambda g: g.intersection(cell_poly))
    sub = sub[ sub.geometry.notna() & (~sub.geometry.is_empty) ].reset_index(drop=True)
    return sub

def _ensure_series(colname, default=None):
    # safe accessor for missing edge columns
    return edges[colname] if colname in edges.columns else pd.Series(default, index=edges.index)

# 1) Sanity: same metric CRS for edges & grid --------------------------------
assert edges.crs == grid.crs, f"CRS mismatch: edges={edges.crs}, grid={grid.crs}"

# 2) Ensure edge-level flags/scores exist (bikeable, obstacles, quality) -----
if "hw" not in edges.columns:
    edges["hw"] = edges["highway"].astype(str)

ROADCLASS_MINOR = {"tertiary","tertiary_link","residential","living_street","unclassified"}
ROADCLASS_PATHS = {"cycleway","path","footway","pedestrian","track","service"}

if "bikeable" not in edges.columns:
    cycle_cols = [c for c in edges.columns if c.startswith("cycleway")]
    cycle_fac_any = edges[cycle_cols].notna().any(axis=1) if cycle_cols else pd.Series(False, index=edges.index)
    edges["bikeable"] = edges["hw"].isin(ROADCLASS_MINOR | ROADCLASS_PATHS) | cycle_fac_any | edges["hw"].eq("cycleway")

# Discomfort proxies -> boolean flags
for tag in ["barrier","traffic_calming","crossing","ford","traffic_signals","signal"]:
    col = f"has_{tag}"
    if col not in edges.columns:
        edges[col] = _ensure_series(tag).notna()

# Quality proxies (surface + width → 0..1, later ×10)
if "surface_quality_score" not in edges.columns:
    surf = _ensure_series("surface", default=np.nan).astype(str)
    edges["surface_quality_score"] = np.select(
        [
            surf.str.contains("asphalt|concrete|paved", case=False, na=False),
            surf.str.contains("compacted|fine_gravel|paving_stones", case=False, na=False),
            surf.str.contains("gravel|unpaved|ground|earth|dirt|cobble|setts|sand", case=False, na=False),
        ],
        [1.0, 0.6, 0.2],
        default=np.nan,
    )

if "width_score" not in edges.columns:
    width = pd.to_numeric(_ensure_series("width"), errors="coerce")
    edges["width_score"] = np.select(
        [width >= 3.0, width >= 2.2, width >= 1.5],
        [1.0, 0.7, 0.4],
        default=np.nan
    )

# Ensure length field exists and drop zero-length
if "length_m" not in edges.columns:
    edges["length_m"] = edges.geometry.length
edges = edges[edges["length_m"] > 0].copy()

# 3) Per-cell computations ----------------------------------------------------
rows = []
for i, cell_poly in enumerate(grid.geometry):
    area_km2 = cell_poly.area / 1e6 if cell_poly.area else 0.0

    # CN_raw: bikeable length density (km per km^2)
    bike_len_m = clip_edges_sum_length(edges, cell_poly, edges["bikeable"])
    cn_raw = 0.0 if area_km2 == 0 else (bike_len_m / 1000.0) / area_km2

    # DISC: share of length WITHOUT obstacles/controls (higher is better)
    total_len = clip_edges_sum_length(edges, cell_poly)
    obs_mask = edges[[c for c in edges.columns if c.startswith("has_")]].any(axis=1)
    with_obs_len = clip_edges_sum_length(edges, cell_poly, obs_mask)
    share_clean = 0.0 if total_len == 0 else 100.0 * (1.0 - min(with_obs_len / total_len, 1.0))

    # QLT: length-weighted mean(surface_quality_score, width_score) in cell
    sub = intersect_edges_in_cell(edges, cell_poly)
    if not sub.empty:
        sub["len"] = sub.length
        sub["qual_pair"] = sub[["surface_quality_score","width_score"]].mean(axis=1, skipna=True)
        lw_avg = float(np.average(sub["qual_pair"].fillna(0.5), weights=sub["len"]))  # fallback 0.5
        qlt = lw_avg * 10.0  # scale to 0..10
    else:
        qlt = np.nan

    rows.append({"cell_id": i, "CN_raw": cn_raw, "DISC_pct": share_clean, "QLT": qlt})

new_df = pd.DataFrame(rows).set_index("cell_id")

# 4) CN: quantile map CN_raw → 0..10
def _quantile_scores(series, k=5):
    s = series.dropna()
    if s.empty:
        return pd.Series(np.nan, index=series.index)
    qs = np.quantile(s, np.linspace(0, 1, k+1))
    qs = np.unique(qs)  # guard against duplicates
    cats = pd.cut(series, bins=qs, include_lowest=True, labels=False)
    return (cats / (k-1) * 10).astype(float)

new_df["CN"] = _quantile_scores(new_df["CN_raw"], k=5)

# 5) DISC: map percent to 0..10 via fixed bins (tweak thresholds if needed)
def _share_to_score(x, bins=(0,20,40,60,80,100), scores=(0,2,4,6,8,10)):
    if pd.isna(x):
        return np.nan
    x = float(np.clip(x, 0, 100))
    idx = np.searchsorted(bins, x, side="right")
    idx = min(idx, len(scores)-1)
    return scores[idx]

new_df["DISC"] = new_df["DISC_pct"].apply(_share_to_score)
new_df = new_df.drop(columns=["CN_raw","DISC_pct"])

# 6) Merge into grid_scores (create if needed) -------------------------------
if "grid_scores" not in globals():
    grid_scores = grid.copy()
    grid_scores["cell_id"] = grid_scores.index.astype(int)

grid_scores = grid_scores.merge(new_df, left_on="cell_id", right_index=True, how="left")

# 7) Preview -----------------------------------------------------------------
grid_scores[["cell_id"] + [c for c in ["CN","DISC","QLT"] if c in grid_scores.columns]].head(10)


,cell_id,CN,DISC,QLT
0,0,0.0,10,5.000000
1,1,0.0,10,5.000000
2,2,0.0,10,5.000000
3,3,0.0,10,5.000000
4,4,0.0,10,5.000000
5,5,0.0,10,5.000000
6,6,0.0,10,5.000000
7,7,2.5,10,5.001145
8,8,0.0,10,5.000000
9,9,0.0,10,5.000000


In [15]:
# === Automatic BI computation block ===
import numpy as np
import pandas as pd

# Define weights for all factors you might have
# (set to 0 if you don't want that factor to count yet)
WEIGHTS = {
    "RC":    0.26,  #Road Class
    "MRSEP": 0.21,  #Main Road Separation
    "CN":    0.13,  #Cycling Network
    "DISC":  0.13,  #Cycling Road Discomfort
    "QLT":   0.13,  #Cyvling Road Comfort
    "TOP":   0.14,  #Topography
    "SOFT":  0.00,  #Soft Factors
    "AESTH": 0.00,  #Aesthetics & Attractiveness
}

# Keep only weights for factor columns that actually exist in grid_scores
present_weights = {k: v for k, v in WEIGHTS.items() if k in grid_scores.columns}

# Normalize weights so they sum to 1
total_w = sum(present_weights.values())
if total_w == 0:
    raise ValueError("All weights are zero or none of the specified factor columns exist in grid_scores.")
present_weights = {k: v / total_w for k, v in present_weights.items()}

# Function to compute weighted BI per row
def weighted_bi(row, w):
    vals, wts = [], []
    for col, wt in w.items():
        val = row[col]
        if pd.notna(val):
            vals.append(val * wt)
            wts.append(wt)
    return np.nan if not vals else sum(vals) / sum(wts)

# Apply function across all rows
grid_scores["BI"] = grid_scores.apply(lambda r: weighted_bi(r, present_weights), axis=1)

# Preview result
grid_scores[["cell_id"] + list(present_weights.keys()) + ["BI"]].head(10)


,cell_id,RC,MRSEP,CN,DISC,QLT,BI
0,0,10,NaN,0.0,10,5.000000,7.000000
1,1,10,NaN,0.0,10,5.000000,7.000000
2,2,8,NaN,0.0,10,5.000000,6.200000
3,3,10,NaN,0.0,10,5.000000,7.000000
4,4,8,2.0,0.0,10,5.000000,5.174419
5,5,10,2.0,0.0,10,5.000000,5.779070
6,6,10,NaN,0.0,10,5.000000,7.000000
7,7,10,2.0,2.5,10,5.001145,6.157150
8,8,10,NaN,0.0,10,5.000000,7.000000
9,9,8,NaN,0.0,10,5.000000,6.200000


In [1]:
# --- Download Copernicus 30m DEM for Munich via OpenTopography API ---
import os, json, requests, geopandas as gpd, osmnx as ox
from pathlib import Path

API_KEY = "dce70f9968e925fd43207861c10a7d19"

# 1) Get Munich boundary in WGS84
munich = ox.geocode_to_gdf("Munich, Germany").to_crs(4326)
poly = munich.geometry.iloc[0]
minx, miny, maxx, maxy = poly.bounds

# 2) Parameters for API request
params = {
    "demtype": "COP30",  # Copernicus GLO-30 DEM
    "south":  miny,
    "north":  maxy,
    "west":   minx,
    "east":   maxx,
    "outputFormat": "GTiff",
    "API_Key": API_KEY
}

url = "https://portal.opentopography.org/API/globaldem"

# 3) Call API and download raster
print("Requesting DEM from OpenTopography...")
r = requests.get(url, params=params, timeout=300)
r.raise_for_status()

# 4) Save file locally
out_dir = Path.cwd() / "dem"
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "munich_cop30.tif"
with open(out_path, "wb") as f:
    f.write(r.content)

print("✅ DEM saved to:", out_path)


Requesting DEM from OpenTopography...
✅ DEM saved to: C:\Users\valeriia.sailaonova\dem\munich_cop30.tif


In [3]:
# --- Re-create (or load) OSM graph G, then compute TOP and BI ---

import os
from pathlib import Path
import numpy as np, pandas as pd, geopandas as gpd, osmnx as ox

# 0) Paths
out_dir = Path.cwd() / "outputs"
out_dir.mkdir(parents=True, exist_ok=True)
graphml = out_dir / "munich_graph.graphml"              # cache for the graph
dem_path = str((Path.cwd() / "dem" / "munich_cop30.tif").resolve())  # your DEM from the previous step

# 1) Ensure we have G (EPSG:4326)
if 'G' not in globals():
    if graphml.exists():
        print("Loading graph from:", graphml)
        G = ox.load_graphml(graphml)
    else:
        print("Downloading OSM graph for Munich…")
        boundary = ox.geocode_to_gdf("Munich, Germany").to_crs(4326)
        poly = boundary.geometry.unary_union
        # Cycling-friendly custom filter: roads + paths, excludes areas/private
        cf = (
            '["highway"~"motorway|trunk|primary|secondary|tertiary|unclassified|'
            'residential|living_street|service|road|busway|cycleway|path|footway|'
            'pedestrian|track|bridleway|steps"]'
            '["area"!~"yes"]["access"!~"private|no"]'
        )
        G = ox.graph_from_polygon(poly, custom_filter=cf, simplify=True)
        ox.save_graphml(G, graphml)
        print("Saved graph to:", graphml)

# 2) Compute slopes from DEM
G2 = ox.add_node_elevations_raster(G, dem_path, band=1)
G2 = ox.add_edge_grades(G2, add_absolute=True)
_, edges_top = ox.graph_to_gdfs(G2, nodes=True, edges=True, fill_edge_geometry=True)

# 3) Project edges to your grid CRS and map slope to 0–10 (flatter=better)
edges_top = edges_top.to_crs(grid.crs)
edges_top["grade_abs"] = pd.to_numeric(edges_top["grade_abs"], errors="coerce")

bins   = [0, 0.01, 0.03, 0.05, 0.08, 1.0]  # 1%, 3%, 5%, 8%+
scores = [10,  8,    6,    4,    2]
def slope_score(g):
    if pd.isna(g): return np.nan
    i = np.searchsorted(bins, g, side="right") - 1
    return scores[max(0, min(i, len(scores)-1))]

edges_top["slope_score"] = edges_top["grade_abs"].map(slope_score)

# 4) Length-weighted average slope score per grid cell -> TOP (0..10)
top_vals = []
for i, cell in enumerate(grid.geometry):
    sub = edges_top.loc[edges_top.geometry.intersects(cell)].copy()
    if sub.empty:
        top_vals.append((i, np.nan)); continue
    sub["geom_clip"] = sub.geometry.apply(lambda g: g.intersection(cell))
    sub = sub[sub["geom_clip"].notna() & (~sub["geom_clip"].is_empty)]
    if sub.empty:
        top_vals.append((i, np.nan)); continue
    w = sub["geom_clip"].length
    val = float(np.average(sub["slope_score"].fillna(5), weights=w))  # fallback 5
    top_vals.append((i, val))

TOP_df = pd.DataFrame(top_vals, columns=["cell_id","TOP"]).set_index("cell_id")
grid_scores = grid_scores.merge(TOP_df, left_on="cell_id", right_index=True, how="left")

# 5) Recompute BI using whatever factors exist now
WEIGHTS = {
    "RC": 0.26, "MRSEP": 0.21, "CN": 0.13, "DISC": 0.13, "QLT": 0.13, "TOP": 0.14,
    "SOFT": 0.00, "AESTH": 0.00
}
present_weights = {k:v for k,v in WEIGHTS.items() if k in grid_scores.columns}
tw = sum(present_weights.values())
present_weights = {k:v/tw for k,v in present_weights.items()} if tw>0 else {}

def weighted_bi(row, w):
    vals=[]; wts=[]
    for col, wt in w.items():
        v = row[col]
        if pd.notna(v):
            vals.append(v*wt); wts.append(wt)
    return np.nan if not vals else sum(vals)/sum(wts)

if present_weights:
    grid_scores["BI"] = grid_scores.apply(lambda r: weighted_bi(r, present_weights), axis=1)

# 6) Save and preview
grid_scores.to_file(out_dir / "bikeability_reference.gpkg",
                    layer="bi_grid", driver="GPKG", engine="fiona", promote_to_multi=True)
print("Saved updated GeoPackage:", out_dir / "bikeability_reference.gpkg")

cols = ["cell_id","RC","MRSEP","CN","DISC","QLT","TOP","BI"]
print(grid_scores[[c for c in cols if c in grid_scores.columns]].head(10))


C:\Users\valeriia.sailaonova\AppData\Local\Temp\ipykernel_30028\4027563975.py:21: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  poly = boundary.geometry.unary_union


Saved graph to: C:\Users\valeriia.sailaonova\outputs\munich_graph.graphml


ImportError: rasterio must be installed as an optional dependency to query rasters.